In [ ]:
!pip install -q transformers
!pip install -q sentencepiece
!pip install -q datasets
!pip install -q accelerate
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.0 MB/s eta 0:00:00


In [ ]:
import os
import pickle
import faiss
import numpy as np
import pandas as pd

from tqdm import tqdm

from datasets import Dataset

from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

from google.colab import drive

In [ ]:
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
DATA="/content/drive/MyDrive/FIFI_Research/data"

train_df=pd.read_csv(
    os.path.join(DATA,"train.tsv"),
    sep="\t"
)

val_df=pd.read_csv(
    os.path.join(DATA,"val.tsv"),
    sep="\t"
)

print(train_df.shape)
print(val_df.shape)

(90000, 4)
(18000, 4)


In [ ]:
MODEL_PATH="/content/drive/MyDrive/FIFI_Research/models"

with open(
    os.path.join(MODEL_PATH,"candidate_titles.pkl"),
    "rb"
) as f:
    candidate_titles=pickle.load(f)

candidate_embeddings=np.load(
    os.path.join(MODEL_PATH,"candidate_embeddings.npy")
)

index=faiss.read_index(
    os.path.join(MODEL_PATH,"faiss.index")
)

print("Candidate Titles :",len(candidate_titles))
print("Embedding Shape :",candidate_embeddings.shape)

Candidate Titles : 2000
Embedding Shape : (2000, 768)


In [ ]:
embedder=SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
expansion_rules={

"image":[
"vision",
"segmentation",
"detection",
"classification",
"cnn"
],

"learning":[
"deep",
"neural",
"representation",
"training",
"model"
],

"network":[
"graph",
"neural",
"architecture",
"learning"
],

"brain":[
"fmri",
"eeg",
"neural",
"cognitive"
],

"robot":[
"robotics",
"navigation",
"planning",
"control"
],

"language":[
"nlp",
"bert",
"transformer",
"text"
],

"video":[
"tracking",
"motion",
"recognition"
],

"detection":[
"recognition",
"classification",
"segmentation"
],

"classification":[
"prediction",
"recognition",
"learning"
]
}

In [ ]:
def expand_query(query):

    query=query.lower()

    extra=[]

    for key in expansion_rules:

        if key in query:

            extra.extend(
                expansion_rules[key]
            )

    extra=list(set(extra))

    return query+" "+" ".join(extra)

In [ ]:
train_df["expanded_query"]=train_df[
    "generated_title"
].apply(expand_query)

val_df["expanded_query"]=val_df[
    "generated_title"
].apply(expand_query)

train_df[
[
"generated_title",
"expanded_query"
]
].head()

,generated_title,expanded_query
0,Optimizing Class-Specific Complex Network Gene...,optimizing class-specific complex network gene...
1,Monte Carlo Self-Supervised Proposal-Based Obj...,monte carlo self-supervised proposal-based obj...
2,Task-Level Planning and Simulation for Robotic...,task-level planning and simulation for robotic...
3,Using Map Search Data to Find the Best Locatio...,using map search data to find the best locatio...
4,Evaluating the Perceptual Capabilities and Cog...,evaluating the perceptual capabilities and cog...


In [ ]:
train_query_embeddings=embedder.encode(

    train_df["expanded_query"].tolist(),

    normalize_embeddings=True,

    batch_size=64,

    show_progress_bar=True
)

val_query_embeddings=embedder.encode(

    val_df["expanded_query"].tolist(),

    normalize_embeddings=True,

    batch_size=64,

    show_progress_bar=True
)

Batches:   0%|          | 0/1407 [00:00<?, ?it/s]

Batches:   0%|          | 0/282 [00:00<?, ?it/s]

In [ ]:
train_scores,train_indices=index.search(

    train_query_embeddings,

    5
)

val_scores,val_indices=index.search(

    val_query_embeddings,

    5
)

In [ ]:
train_candidates=[]

for row in train_indices:

    train_candidates.append(

        [

            candidate_titles[idx]

            for idx in row

        ]

    )

val_candidates=[]

for row in val_indices:

    val_candidates.append(

        [

            candidate_titles[idx]

            for idx in row

        ]

    )

In [ ]:
RAG_PATH="/content/drive/MyDrive/FIFI_Research/models"

with open(
    os.path.join(RAG_PATH,"train_candidates.pkl"),
    "wb"
) as f:

    pickle.dump(train_candidates,f)

with open(
    os.path.join(RAG_PATH,"val_candidates.pkl"),
    "wb"
) as f:

    pickle.dump(val_candidates,f)

print("RAG candidates saved successfully.")

RAG candidates saved successfully.


In [ ]:
i=0

print("="*100)

print("Generated Title:\n")
print(train_df.loc[i,"generated_title"])

print("\n")

print("Ground Truth:\n")
print(train_df.loc[i,"original_title"])

print("\n")

print("Retrieved Top-5:\n")

for j,title in enumerate(train_candidates[i],1):

    print(j,title)

Generated Title:

Optimizing Class-Specific Complex Network Generation for High-Level Classification Using Genetic Algorithms


Ground Truth:

Improve High Level Classification with a More Sensitive metric and  Optimization approach for Complex Network Building


Retrieved Top-5:

1 Dynamic Optimization of Neural Network Structures Using Probabilistic  Modeling
2 Adversarial Genetic Programming for Cyber Security: A Rising Application  Domain Where GP Matters
3 Kernel-Based Training of Generative Networks
4 Multi-Objective Neural Architecture Search Based on Diverse Structures  and Adaptive Recommendation
5 BEAR: Sketching BFGS Algorithm for Ultra-High Dimensional Feature  Selection in Sublinear Memory


In [ ]:
def build_rag_prompt(row, candidates):

    prompt = f"""Recover the ORIGINAL scientific paper title.

Style:
{row['category']}

Rewritten Title:
{row['generated_title']}

Top Candidate Original Titles:

"""

    for i, title in enumerate(candidates, 1):

        prompt += f"{i}. {title}\n"

    prompt += "\nReturn ONLY the original paper title."

    return prompt

In [ ]:
train_prompts = []

for i in tqdm(range(len(train_df))):

    train_prompts.append(

        build_rag_prompt(

            train_df.iloc[i],

            train_candidates[i]

        )

    )

val_prompts = []

for i in tqdm(range(len(val_df))):

    val_prompts.append(

        build_rag_prompt(

            val_df.iloc[i],

            val_candidates[i]

        )

    )

train_df["input_text"] = train_prompts

val_df["input_text"] = val_prompts

train_df["target_text"] = train_df["original_title"]

val_df["target_text"] = val_df["original_title"]

100%|██████████| 18000/18000 [00:00<00:00, 21104.41it/s]


In [ ]:
print(train_df["input_text"][0])

print()

print("="*100)

print()

print(train_df["target_text"][0])

Recover the ORIGINAL scientific paper title.

Style:
technical

Rewritten Title:
Optimizing Class-Specific Complex Network Generation for High-Level Classification Using Genetic Algorithms

Top Candidate Original Titles:

1. Dynamic Optimization of Neural Network Structures Using Probabilistic  Modeling
2. Adversarial Genetic Programming for Cyber Security: A Rising Application  Domain Where GP Matters
3. Kernel-Based Training of Generative Networks
4. Multi-Objective Neural Architecture Search Based on Diverse Structures  and Adaptive Recommendation
5. BEAR: Sketching BFGS Algorithm for Ultra-High Dimensional Feature  Selection in Sublinear Memory

Return ONLY the original paper title.


Improve High Level Classification with a More Sensitive metric and  Optimization approach for Complex Network Building


In [ ]:
train_dataset = Dataset.from_pandas(

    train_df[
        [
            "input_text",
            "target_text"
        ]
    ]

)

val_dataset = Dataset.from_pandas(

    val_df[
        [
            "input_text",
            "target_text"
        ]
    ]

)

print(train_dataset)

print(val_dataset)

Dataset({
    features: ['input_text', 'target_text'],
    num_rows: 90000
})
Dataset({
    features: ['input_text', 'target_text'],
    num_rows: 18000
})


In [ ]:
MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
MAX_INPUT = 256

MAX_OUTPUT = 64

In [ ]:
def preprocess(examples):

    inputs = tokenizer(

        examples["input_text"],

        max_length=MAX_INPUT,

        truncation=True

    )

    labels = tokenizer(

        examples["target_text"],

        max_length=MAX_OUTPUT,

        truncation=True

    )

    inputs["labels"] = labels["input_ids"]

    return inputs

In [ ]:
train_dataset = train_dataset.map(

    preprocess,

    batched=True,

    remove_columns=train_dataset.column_names

)

val_dataset = val_dataset.map(

    preprocess,

    batched=True,

    remove_columns=val_dataset.column_names

)

Map:   0%|          | 0/90000 [00:00<?, ? examples/s]

Map:   0%|          | 0/18000 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorForSeq2Seq(

    tokenizer,

    model=model

)

In [ ]:
training_args = Seq2SeqTrainingArguments(

    output_dir="/content/RAG_Task2",

    learning_rate=3e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    gradient_accumulation_steps=2,

    num_train_epochs=3,

    predict_with_generate=True,

    save_strategy="epoch",

    eval_strategy="epoch",

    logging_steps=100,

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    greater_is_better=False,

    fp16=True,

    save_total_limit=1,

    report_to="none"
)

In [ ]:
trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    data_collator=data_collator
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.000000,nan
2,0.000000,nan


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,0.000000,nan
2,0.000000,nan
3,0.000000,nan


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=16875, training_loss=0.0, metrics={'train_runtime': 8272.3063, 'train_samples_per_second': 32.639, 'train_steps_per_second': 2.04, 'total_flos': 6.390418816652083e+16, 'train_loss': 0.0, 'epoch': 3.0})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
SAVE_PATH="/content/drive/MyDrive/FIFI_Research/models/flan_rag"

model.save_pretrained(SAVE_PATH)

tokenizer.save_pretrained(SAVE_PATH)

print("RAG Model Saved Successfully")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RAG Model Saved Successfully


In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)

model = AutoModelForSeq2SeqLM.from_pretrained(SAVE_PATH)

import torch

device=torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

model.eval()

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [ ]:
from tqdm import tqdm

batch_size=16

predictions=[]

for i in tqdm(range(0,len(val_df),batch_size)):

    batch=val_df["input_text"].iloc[i:i+batch_size].tolist()

    inputs=tokenizer(

        batch,

        padding=True,

        truncation=True,

        return_tensors="pt"

    ).to(device)

    outputs=model.generate(

        **inputs,

        max_new_tokens=40,

        num_beams=5,

        early_stopping=True,

        no_repeat_ngram_size=3,

        repetition_penalty=1.2,

        length_penalty=1.0

    )

    preds=tokenizer.batch_decode(

        outputs,

        skip_special_tokens=True

    )

    predictions.extend(preds)

100%|██████████| 1125/1125 [38:48<00:00,  2.07s/it]


In [ ]:
val_df["prediction"]=predictions

val_df[
[
"generated_title",
"prediction",
"original_title"
]
].head(20)

,generated_title,prediction,original_title
0,Fully Convolutional Joint Detection and Regres...,Fully Convolutional Joint Detection and Regres...,Preterm infants' limb-pose estimation from dep...
1,When Machines Argue: Teaching AI to Spot Fake ...,When Machines Argue: Teaching AI to Spot Fake ...,DSGAN: Generative Adversarial Training for Dis...
2,Optimized 2D Manifold Folding and Attribute Ma...,Optimized 2D Manifold Folding and Attribute Ma...,Folding-based compression of point cloud attri...
3,From Snapshot to Sawdust: Rebuilding Wooden Ob...,From Snapshot to Sawdust: Rebuilding Wooden Ob...,Fabrication-Aware Reverse Engineering for Carp...
4,"A Survey of How Deep Learning Improves Image, ...","A Survey of How Deep Learning Improves Image, ...",Super-Resolution via Deep Learning
5,A Toolkit for Building Self-Navigating Robots:...,A Toolkit for Building Self-Navigating Robots:...,Autonomous Exploration Development Environment...
6,How to Share Machine Learning Resources Fairly...,How to Share Machine Learning Resources Fairly...,Ease.ml: Towards Multi-tenant Resource Sharing...
7,Clearing Up Shaky Video: How to Remove Atmosph...,Clearing Up Shaky Video: How to Remove Atmosph...,Atmospheric turbulence mitigation for sequence...
8,Educational Data Mining with Logistic Regressi...,Educational Data Mining with Logistic Regressi...,Modeling the EdNet Dataset with Logistic Regre...
9,How Do AI Language Models Learn Words? Compari...,How Do AI Language Models Learn Words? Compari...,Word Acquisition in Neural Language Models


In [ ]:
from collections import Counter
import numpy as np

def token_f1(pred,gt):

    pred=pred.lower().split()

    gt=gt.lower().split()

    common=Counter(pred)&Counter(gt)

    overlap=sum(common.values())

    if overlap==0:

        return 0

    precision=overlap/len(pred)

    recall=overlap/len(gt)

    return 2*precision*recall/(precision+recall)

scores=[]

for pred,gt in zip(

    val_df["prediction"],

    val_df["original_title"]

):

    scores.append(

        token_f1(pred,gt)

    )

print("Mean Token F1 =",np.mean(scores))

Mean Token F1 = 0.2651626332194824


In [ ]:
for style in [

"technical",

"accessible",

"catchy"

]:

    subset=val_df[

        val_df["category"]==style

    ]

    style_scores=[]

    for pred,gt in zip(

        subset["prediction"],

        subset["original_title"]

    ):

        style_scores.append(

            token_f1(pred,gt)

        )

    print(

        style,

        np.mean(style_scores)

    )

technical 0.4098351282220965
accessible 0.18259665761670382
catchy 0.20305611381964703


In [ ]:
submission=val_df[

[
"id",
"category",
"generated_title"

]

].copy()

submission["original_title"]=predictions

submission.to_csv(

"/content/drive/MyDrive/FIFI_Research/submissions/FutureMinds_task2_run2.tsv",

sep="\t",

index=False

)

print(submission.head())

   id    category                                    generated_title  \
0   0   technical  Fully Convolutional Joint Detection and Regres...   
1   1      catchy  When Machines Argue: Teaching AI to Spot Fake ...   
2   2   technical  Optimized 2D Manifold Folding and Attribute Ma...   
3   3      catchy  From Snapshot to Sawdust: Rebuilding Wooden Ob...   
4   4  accessible  A Survey of How Deep Learning Improves Image, ...   

                                      original_title  
0  Fully Convolutional Joint Detection and Regres...  
1  When Machines Argue: Teaching AI to Spot Fake ...  
2  Optimized 2D Manifold Folding and Attribute Ma...  
3  From Snapshot to Sawdust: Rebuilding Wooden Ob...  
4  A Survey of How Deep Learning Improves Image, ...  
